In [30]:
#import packages
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

cases_df = pd.read_csv("weekly AFR cases by country as of 19 January 2025(in).csv")
print(cases_df.head())

  country iso3 week_end_date  total_confirmed_cases  total_suspected_cases  \
0  Angola  AGO    12/29/2024                      4                      0   
1  Angola  AGO    12/22/2024                      4                      0   
2  Angola  AGO    12/15/2024                      4                      0   
3  Angola  AGO     12/8/2024                      4                      0   
4  Angola  AGO     12/1/2024                      3                      0   

   total_deaths  total_suspected_deaths  new_confirmed_cases  \
0             0                       0                    0   
1             0                       0                    0   
2             0                       0                    0   
3             0                       0                    1   
4             0                       0                    1   

   new_suspected_cases  new_deaths  new_suspected_deaths  
0                    0           0                     0  
1                    0      

In [32]:
#cleaning cases data
# Select relevant columns
cases_df_model = cases_df[['country', 'week_end_date', 'total_confirmed_cases', 'new_confirmed_cases']].copy()
cases_df_model['week_end_date'] = pd.to_datetime(cases_df_model['week_end_date'])

cases_df_model = cases_df_model.sort_values(by=['country', 'week_end_date'], ascending=[True, True])

# Create lag feature for new weekly cases
cases_df_model['previous_week_cases'] = cases_df_model.groupby('country', group_keys=False)['total_confirmed_cases'].shift(1)

# Update column safely without using inplace=True
cases_df_model['previous_week_cases'] = cases_df_model['previous_week_cases'].fillna(0)

cases_df_model['month'] = cases_df_model['week_end_date'].dt.month  # seasonality effect

cases_df_model.iloc[170:191]

,country,week_end_date,total_confirmed_cases,new_confirmed_cases,previous_week_cases,month
183,Burundi,2024-09-01,328,97,231.0,9
182,Burundi,2024-09-08,385,57,328.0,9
181,Burundi,2024-09-15,564,179,385.0,9
180,Burundi,2024-09-22,696,132,564.0,9
179,Burundi,2024-09-29,853,157,696.0,9
178,Burundi,2024-10-06,987,134,853.0,10
177,Burundi,2024-10-13,1170,183,987.0,10
176,Burundi,2024-10-20,1287,117,1170.0,10
175,Burundi,2024-10-27,1509,222,1287.0,10
174,Burundi,2024-11-03,1718,209,1509.0,11


In [ ]:
healthcare_df = pd.read_csv("Healthcare expenditure data as percentage of GDP (%).csv")
healthcare_df.head()

In [ ]:
#testing cases data

# Step 1: Have a data frame with relevant factors
df = cases_df_model.copy()

# Convert week_end_date to numerical format
df['week_end_date_num'] = df['week_end_date'].map(pd.Timestamp.toordinal)

# Step 2: Define features (x) and target (y)
x = df[['week_end_date_num', 'total_confirmed_cases','previous_week_cases', 'new_confirmed_cases', 'month']]
y = df['new_confirmed_cases']

# Step 3: Split data into training (80%) and testing (20%)
n = 10
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=n)

# Step 4: Train the Random Forest Model
rf = RandomForestRegressor(n_estimators=100, random_state=n)
rf.fit(x_train, y_train)

# Step 5: Compute Feature Importance
feature_importance = pd.DataFrame({
    'Feature': x.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("Feature Importance:")
print(feature_importance)

# Step 6: Make Predictions on Test Data
y_pred = rf.predict(x_test)

# Step 7: Evaluate Model
mae = mean_absolute_error(y_test, y_pred)
print(f'Mean Absolute Error: {mae:.2f}')

# Step 8: Predict Future (e.g. 2025) Monkeypox Cases
# function to predict
def predict_cases_for_date(last_known_date, total_cases_so_far, last_week_cases, future_date):
    last_known_date = pd.to_datetime(last_known_date)
    future_date = pd.to_datetime(future_date)
    
    if future_date <= last_known_date:
        print("Error: Future date must be after the last known date.")
        return None
    
    num_weeks = (future_date - last_known_date).days // 7
    
    predicted_total_cases = total_cases_so_far
    previous_week_cases = last_week_cases
    current_date = last_known_date

    for i in range(num_weeks):
        current_date_num = current_date.toordinal()
        month = current_date.month
        new_confirmed_cases = predicted_total_cases - previous_week_cases
        
        new_data = pd.DataFrame({
            'week_end_date_num': [current_date_num],
            'total_confirmed_cases': [predicted_total_cases],
            'previous_week_cases': [previous_week_cases],
            'new_confirmed_cases': [new_confirmed_cases],
            'month': [month],
        })
        
        # Predict this week's total cases
        predicted_new_cases = rf.predict(new_data)[0]
        
        # Update for the next iteration
        previous_week_cases = predicted_total_cases
        predicted_total_cases += predicted_new_cases
        current_date += pd.Timedelta(weeks=1)
    
    print(f"Predicted Monkeypox Cases for {future_date.date()}: {predicted_total_cases:.0f}")

Feature Importance:
                 Feature  Importance
3    new_confirmed_cases    0.998718
1  total_confirmed_cases    0.000394
2    previous_week_cases    0.000363
4                  month    0.000283
0      week_end_date_num    0.000242
Mean Absolute Error: 0.28


In [38]:
predict_cases_for_date('2025-01-05', 3035, 2946, '2025-01-12')

Predicted Monkeypox Cases for 2025-01-12: 3126


In [ ]:
# Step 1: Have a data frame with data of the different factors we want to look at
# e.g. df = pd.DataFrame()

# (Step 1.5: Could create lag features for previous year, for the model to use previous year data as another factor)
# e.g. df['Lag_Healthcare_Expenditure'] = df['Healthcare_Expenditure'].shift(1)

# Step 2: Define factors (x) and target (y)
# e.g. x = df[['Year', 'Lag_Healthcare_Expenditure', 'Population_Density']]
#      y = df['Monkeypox_Cases']

# Step 4: Split data into training and testing (e.g. 80% train, 20% test)
# n = 10
# x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=n)
# random state to make sure results are repeatable

# Step 5: Train the Random Forest Model
# e.g. rf = RandomForestRegressor(n_estimators=100, random_state=n)
# rf.fit(x_train, y_train)

# Step 6: Compute Feature Importance
# feature_importance = pd.DataFrame({
#     'Feature': x.columns,
#     'Importance': rf.feature_importances_
# }).sort_values(by='Importance', ascending=False)

# Display Feature Importance
# print("Feature Importance:")
# print(feature_importance)

# Step 7: Make Predictions
# y_pred = rf.predict(x_test)

# Step 8: Evaluate the Model
# mae = mean_absolute_error(y_test, y_pred)
# print(f'Mean Absolute Error: {mae:.2f}')

# Step 9: Predict Future (e.g. 2025) Monkeypox Cases
# e.g. new_data = pd.DataFrame({
#     'Year': [2025],
#     'Lag_Healthcare_Expenditure': [5000],  # Previous year's value (2024)
#     'Lag_Population_Density': [150],  # Previous year's value (2024)
# })

# prediction_2025 = rf.predict(new_data)
# print(f'Predicted Monkeypox Cases for 2025: {prediction_2025[0]:.0f}')